## Step 4 — Trunk Diameter Estimation

This notebook estimates trunk diameter from the segmented trunk masks and corresponding depth images.

The workflow performs the following tasks:

1. Retrieve the RealSense camera intrinsic parameters.
2. Scale the camera intrinsics to match the processed image dimensions.
3. Convert segmented trunk pixels from the depth image into a 3D point cloud.
4. Apply point cloud cleaning steps, including statistical outlier removal and DBSCAN clustering.
5. Extract trunk width from a horizontal slice of the point cloud at a fixed target height above the trunk base.
6. Save per-image diameter estimates and optional point cloud files for inspection.

The resulting output is a CSV file containing raw trunk width estimates and smoothed trunk width estimates for each image.

In [ ]:
# Import Libraries
import os
import json
import glob

import cv2
import numpy as np
import open3d as o3d
import pandas as pd

from tqdm import tqdm
from sklearn.cluster import DBSCAN
from scipy.signal import savgol_filter

### Retrieve Camera Intrinsics

This section retrieves the RealSense depth camera intrinsics (`fx`, `fy`, `cx`, and `cy`) using the Intel RealSense SDK. These parameters are required for converting image pixels and depth values into real-world 3D coordinates.

In [5]:
import pyrealsense2 as rs

# Initialize RealSense pipeline
pipeline = rs.pipeline()
config = rs.config()

# Enable depth stream at 1280 x 720 resolution
config.enable_stream(rs.stream.depth, 1280, 720, rs.format.z16, 30)

# Start streaming to get camera intrinsics
pipeline.start(config)
profile = pipeline.get_active_profile()
depth_stream = profile.get_stream(rs.stream.depth)
intrinsics = depth_stream.as_video_stream_profile().get_intrinsics()

# Extract intrinsic parameters
fx = intrinsics.fx
fy = intrinsics.fy
cx = intrinsics.ppx
cy = intrinsics.ppy

print("Camera Intrinsics:")
print(f"fx: {fx}, fy: {fy}")
print(f"cx: {cx}, cy: {cy}")

# Stop the pipeline after getting the intrinsics
pipeline.stop()

Camera Intrinsics:
fx: 646.3688354492188, fy: 646.3688354492188
cx: 641.2626953125, cy: 363.6773986816406


### Diameter Estimation

This section processes the SAM2 segmentation masks and depth images to estimate trunk diameter.

For each image, the segmented trunk region is converted into a 3D point cloud using the scaled camera intrinsics. The point cloud is then cleaned and filtered, and trunk width is calculated from a narrow horizontal slice at a fixed target height above the trunk base.

Outputs include:

- `.ply` point cloud files for each processed image
- a CSV file containing raw and smoothed trunk width estimates

In [ ]:
import os
import json
import numpy as np
import open3d as o3d
import cv2
import pandas as pd

from scipy.signal import savgol_filter
from sklearn.cluster import DBSCAN
from tqdm import tqdm


# Configuration
depth_folder_base = "path/to/depth_folder"
batch_prefix = "SAM2_mask_data_batch_"
output_csv_path = os.path.join(depth_folder_base, "widths_smoothed.csv")
pointcloud_folder = os.path.join(depth_folder_base, "PointCloud")
os.makedirs(pointcloud_folder, exist_ok=True)

# Manual input for original stream resolution
stream_width = 1280
stream_height = 720

# Original intrinsics for 1280 x 720 stream resolution
fx_raw_stream = 651.337646484375
fy_raw_stream = 651.337646484375
cx_raw_stream = 643.326171875
cy_raw_stream = 361.100830078125

# Dimensions of processed images
input_width = 1280
input_height = 495


def scale_intrinsics(fx_raw, fy_raw, cx_raw, cy_raw, from_size, to_size):
    from_width, from_height = from_size
    to_width, to_height = to_size

    scale_x = to_width / from_width
    scale_y = to_height / from_height

    fx = fx_raw * scale_x
    fy = fy_raw * scale_y
    cx = cx_raw * scale_x
    cy = cy_raw * scale_y

    return {
        "fx": fx,
        "fy": fy,
        "cx": cx,
        "cy": cy,
        "width": to_width,
        "height": to_height
    }


# Scale intrinsics for processed image size
from_size = (stream_width, stream_height)
to_size = (input_width, input_height)

intrinsics = scale_intrinsics(
    fx_raw_stream,
    fy_raw_stream,
    cx_raw_stream,
    cy_raw_stream,
    from_size,
    to_size
)

fx, fy = intrinsics["fx"], intrinsics["fy"]
cx, cy = intrinsics["cx"], intrinsics["cy"]

print(f"Scaled Intrinsics for {input_width}x{input_height}:")
print(f"fx: {fx:.4f}, fy: {fy:.4f}, cx: {cx:.4f}, cy: {cy:.4f}")


# Parameters
target_height = 0.4
tolerance = 0.01
savgol_window = 5
savgol_polyorder = 2
outlier_nb_neighbors = 30
outlier_std_ratio = 1.0
dbscan_eps = 0.02
dbscan_min_samples = 10


def generate_point_cloud(depth_image, mask_array, rgb_image):
    height, width = depth_image.shape
    points, colors = [], []

    for y in range(height):
        for x in range(width):
            if mask_array[y, x] > 0:
                z = depth_image[y, x] / 1000.0

                if z == 0:
                    continue

                x_world = (x - cx) * z / fx
                y_world = (y - cy) * z / fy

                points.append([x_world, y_world, z])
                colors.append(rgb_image[y, x, :] / 255.0)

    if not points:
        return None

    pc = o3d.geometry.PointCloud()
    pc.points = o3d.utility.Vector3dVector(points)
    pc.colors = o3d.utility.Vector3dVector(colors)

    return pc


def flip_point_cloud(pc):
    flip_transform = np.array([
        [-1,  0,  0, 0],
        [ 0, -1,  0, 0],
        [ 0,  0, -1, 0],
        [ 0,  0,  0, 1]
    ])

    pc.transform(flip_transform)
    return pc


def remove_statistical_outliers(pc, nb_neighbors, std_ratio):
    pc, _ = pc.remove_statistical_outlier(
        nb_neighbors=nb_neighbors,
        std_ratio=std_ratio
    )
    return pc


def apply_dbscan(pc, eps, min_samples):
    pts = np.asarray(pc.points)

    if pts.shape[0] == 0:
        return pc

    labels = DBSCAN(eps=eps, min_samples=min_samples).fit_predict(pts)

    if np.max(labels) < 0:
        return pc

    largest_cluster = np.argmax(np.bincount(labels[labels >= 0]))
    indices = np.where(labels == largest_cluster)[0]

    return pc.select_by_index(indices)


def extract_width(points, target_height_m, tolerance_m):
    if points.shape[0] == 0:
        return np.nan

    base_height = np.min(points[:, 1])
    target_rel = target_height_m + base_height

    mask = (
        (points[:, 1] > (target_rel - tolerance_m)) &
        (points[:, 1] < (target_rel + tolerance_m))
    )

    slice_points = points[mask]

    if slice_points.size == 0:
        return np.nan

    x_vals = slice_points[:, 0]

    q1, q3 = np.percentile(x_vals, [25, 75])
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    cleaned_x = x_vals[(x_vals >= lower_bound) & (x_vals <= upper_bound)]

    if cleaned_x.size == 0:
        return np.nan

    width_m = np.max(cleaned_x) - np.min(cleaned_x)
    return width_m * 100.0


# Batch processing
batch_index = 0
all_results = []

while True:
    json_file_path = os.path.join(
        depth_folder_base,
        f"{batch_prefix}{batch_index}.json"
    )

    if not os.path.exists(json_file_path):
        if batch_index == 0:
            print(f"Error: Starting batch file not found: {json_file_path}")
        break

    print(f"\nProcessing batch {batch_index}")

    try:
        with open(json_file_path, "r") as f:
            mask_data = json.load(f)
    except Exception as e:
        print(f"Warning: Skipping batch {batch_index} due to error: {e}")
        batch_index += 1
        continue

    all_images_in_batch = sorted(mask_data.keys())
    print("Image_Name,Raw_Width_cm")

    for image_name in tqdm(all_images_in_batch, desc=f"Batch {batch_index}", unit="image"):
        rgb_path = os.path.join(depth_folder_base, image_name)
        depth_path = rgb_path.replace(".png", ".npy")
        width_cm_val = np.nan

        if not os.path.exists(rgb_path) or not os.path.exists(depth_path):
            continue

        if image_name not in mask_data or not mask_data[image_name]:
            continue

        try:
            rgb_image = cv2.imread(rgb_path)
            if rgb_image is None:
                continue

            depth_image = np.load(depth_path)
            mask_array = np.array(mask_data[image_name][0], dtype=np.uint8)

            pc = generate_point_cloud(depth_image, mask_array, rgb_image)

            if pc and pc.has_points():
                pc = flip_point_cloud(pc)
                pc = remove_statistical_outliers(
                    pc,
                    outlier_nb_neighbors,
                    outlier_std_ratio
                )
                pc = apply_dbscan(pc, dbscan_eps, dbscan_min_samples)

                ply_name = os.path.splitext(image_name)[0] + ".ply"
                ply_path = os.path.join(pointcloud_folder, ply_name)
                o3d.io.write_point_cloud(ply_path, pc)

                points_np = np.asarray(pc.points)
                width_cm_val = extract_width(points_np, target_height, tolerance)

                print(f"{image_name},{width_cm_val:.4f}")

        except Exception as e:
            print(f"Error processing image {image_name}: {e}")

        all_results.append((image_name, width_cm_val))

    batch_index += 1


# Final output
print(f"Processed {len(all_results)} images across {batch_index} batches.")

if all_results:
    df = pd.DataFrame(all_results, columns=["Image_Name", "Raw_Width_cm"])

    raw_widths_cm = df["Raw_Width_cm"].to_numpy()
    smoothed_widths_cm = np.full_like(raw_widths_cm, np.nan)

    valid_indices = np.flatnonzero(~np.isnan(raw_widths_cm))

    if len(valid_indices) >= savgol_window:
        valid_raw = raw_widths_cm[valid_indices]
        smoothed_valid = savgol_filter(valid_raw, savgol_window, savgol_polyorder)
        smoothed_widths_cm[valid_indices] = smoothed_valid

        condition = smoothed_widths_cm < 0
        df["Smoothed_Width_cm"] = np.where(
            condition,
            raw_widths_cm,
            smoothed_widths_cm
        )

    else:
        df["Smoothed_Width_cm"] = raw_widths_cm.copy()

    df.to_csv(output_csv_path, index=False, float_format="%.4f")
    print(f"Saved results to: {output_csv_path}")

else:
    print("No results generated.")

### Optional: Visualize Generated Point Clouds

This section allows interactive visualization of the point cloud files generated during the trunk diameter estimation process.

Each processed image produces a `.ply` file containing the filtered trunk point cloud. These files can be inspected to verify that trunk segmentation and point cloud filtering performed as expected.

This step is optional and intended primarily for quality control and debugging.

In [ ]:
# Define the directory containing generated point clouds
point_cloud_folder = "path/to/PointCloud"

def visualize_point_cloud(ply_path):
    # Load the point cloud
    pcd = o3d.io.read_point_cloud(ply_path)

    # Display the point cloud
    o3d.visualization.draw_geometries([pcd])


# List available point cloud files
ply_files = [f for f in os.listdir(point_cloud_folder) if f.endswith(".ply")]

print("Available point cloud files:")
for i, f in enumerate(ply_files):
    print(f"{i}: {f}")


# Select a point cloud file to visualize
ply_index = int(input("\nEnter the index of the point cloud file you want to visualize: "))

ply_path = os.path.join(point_cloud_folder, ply_files[ply_index])

visualize_point_cloud(ply_path)